# Gene sets

The gene sets Figure 4c tests gPS against: constraint quartiles, cancer drivers, mouse
knockout mortality, developmental disorder panels, FUSIL categories, drug-target and safety
categories, and the reference sets of all genes and all disease-associated genes.

The set membership file is downloaded with the rest of the data. Its upstream construction
(from MGI, FUSIL, DepMap, COSMIC, OMIM, Orphanet and ChEMBL) is in
`chapters/_legacy/04_making_the_list_of_gene_categories` and is not part of this pipeline;
see GAPS.md.

Writes `gene_sets`.

In [1]:
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds

from manuscript_methods import paper

sets = (
    ds.dataset(paper.baseline("list_of_genes_32_categories"), format="parquet")
    .to_table()
    .to_pandas()
    .rename(columns={"targetId": "geneId", "source": "geneSet"})
)

# The human-knockout set is not in that file: it is the canonical transcripts of the knockout
# carriers reported by Minikel et al. 2024 (supplementary table 8).
knockouts = (
    ds.dataset(str(paper.ROOT / "data/41586_2024_7556_MOESM8_ESM.csv"), format="csv")
    .to_table(columns=["TranscriptId"])
    .to_pandas()
)
transcripts = (
    ds.dataset(paper.release("target"), format="parquet")
    .to_table(
        columns={
            "geneId": ds.field("id"),
            "TranscriptId": pc.struct_field(ds.field("canonicalTranscript"), "id"),
        }
    )
    .to_pandas()
)
human_ko = transcripts.merge(knockouts, on="TranscriptId")[["geneId"]].assign(geneSet="human_ko")
print("human knockout genes:", human_ko["geneId"].nunique())

sets = pd.concat([sets, human_ko], ignore_index=True).drop_duplicates()
sets.to_parquet(paper.derived("gene_sets"), index=False)
print("gene-set memberships:", len(sets), "sets:", sets["geneSet"].nunique())
sets["geneSet"].value_counts().to_frame()

human knockout genes: 4445
gene-set memberships: 86855 sets: 33


,count
geneSet,
All genes,20083
all_diseases,8285
lof_constr_Q2,4532
lof_constr_Q1,4526
lof_constr_Q4,4520
lof_constr_Q3,4519
human_ko,4445
omim,4182
hc_paralog,4173
